# Series y contexto

**Capítulo 5 · Universidad de las Hespérides**

Adaptación al español de *Dive into Deep Learning*, Aston Zhang, Zachary C. Lipton, Mu Li y Alexander J. Smola.
Fuente: `locked/chapter_recurrent-neural-networks/sequence.ipynb` · [Lección original](https://d2l.ai/chapter_recurrent-neural-networks/sequence.html).
Texto adaptado bajo [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). [Procedencia y cambios](../PROCEDENCIA.md).
Se conserva la secuencia de las celdas y de los ejercicios; las notas de Hespérides se identifican expresamente.

**Entorno:** ejecuta `uv sync` en la raíz y selecciona su Python como kernel. Las descargas se realizan una vez y quedan en `data/`.
Por defecto, el soporte limita los entrenamientos de `Trainer` a tres épocas y 1024/256 ejemplos para CPU.
Para repetir el régimen completo, inicia Jupyter con `HESPERIDES_COMPLETO=1`. Los ejemplos visuales pequeños conservan su propia configuración explícita.
Los datos de texto en inglés o francés son entradas de los experimentos originales y mantienen su idioma.


In [ ]:
from pathlib import Path
import sys
RAIZ = Path.cwd() if (Path.cwd() / "laboratorio").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from laboratorio import d2l, configurar, epocas
configurar()


# Trabajar con secuencias
<a id="sec_sequence"></a>

Hasta ahora, nos hemos centrado en modelos cuyas entradas consistían en un único vector $\mathbf{x} \in \mathbb{R}^d$. El principal cambio de perspectiva al desarrollar modelos capaces de procesar secuencias es que ahora nos centramos en entradas que consisten en una lista ordenada de vectores de características $\mathbf{x}_1, \dots, \mathbf{x}_T$, donde cada vector de características $\mathbf{x}_t$ está indexado por un paso de tiempo $t \in \mathbb{Z}^+$ pertenece a $\mathbb{R}^d$.

Algunos conjuntos de datos consisten en una sola secuencia masiva. Considere, por ejemplo, las corrientes extremadamente largas de lecturas de sensores que podrían estar disponibles para los científicos climáticos. En tales casos, podríamos crear conjuntos de datos de entrenamiento mediante secuencias de muestreo aleatorio de alguna longitud predeterminada. Más a menudo, nuestros datos llegan como una colección de secuencias. Considere los siguientes ejemplos: (i) una colección de documentos, cada uno representado como su propia secuencia de palabras, y cada uno con su propia longitud $T_i$; (ii) representación de secuencias de estancias de pacientes en el hospital, donde cada estancia consiste en un número de eventos y la longitud de secuencia depende aproximadamente de la duración de la estancia.

Anteriormente, al tratar con insumos individuales, asumimos que fueron muestreados independientemente de la misma distribución subyacente $P(X)$. Si bien seguimos asumiendo que secuencias enteras (por ejemplo, documentos completos o trayectorias del paciente) son muestreadas independientemente, no podemos asumir que los datos que llegan en cada paso del tiempo son independientes entre sí. Por ejemplo, las palabras que probablemente aparecerán más tarde en un documento dependen en gran medida de las palabras que ocurran antes en el documento. El medicamento que un paciente es probable que reciba el décimo día de una visita al hospital depende en gran medida de lo que ocurrió en los nueve días anteriores.

Esto no debería sorprendernos. Si no creyéramos que los elementos de una secuencia estaban relacionados, no nos habríamos molestado en modelarlos como una secuencia en primer lugar. Consideremos la utilidad de las características de auto-llenado que son populares en las herramientas de búsqueda y los clientes de correo electrónico modernos. Son útiles precisamente porque a menudo es posible predecir (imperfecto, pero mejor que adivinar al azar) cuáles podrían ser las probables continuacións de una secuencia, dado algún prefijo inicial. Para la mayoría de los modelos de secuencia, no necesitamos independencia, o incluso estacionalidad, de nuestras secuencias. En cambio, solo necesitamos que las secuencias mismas se muestren de alguna distribución subyacente fija sobre secuencias enteras.

Este enfoque flexible permite fenómenos tales como i) documentos con un aspecto significativamente diferente al principio que al final; o ii) el estado del paciente evolucionando hacia la recuperación o hacia la muerte durante una estancia hospitalaria; o iii) el gusto del cliente evolucionando de manera predecible durante el curso de una interacción continua con un sistema de recomendación.

A veces deseamos predecir un objetivo fijo $y$ dado entrada secuencialmente estructurada (por ejemplo, clasificación de sentimientos basada en una revisión de película). En otras ocasiones, queremos predecir un objetivo secuencialmente estructurado ($y_1, \ldots, y_T$) dado una entrada fija (por ejemplo, subtítulos de imagen). Además, nuestro objetivo es predecir objetivos secuencialmente estructurados basados en entradas secuencialmente estructuradas (por ejemplo, traducción automática o subtítulos de vídeo). Tales tareas secuenciales toman dos formas: i) * alineadas*, donde la entrada en cada paso del tiempo se alinea con un objetivo correspondiente (por ejemplo, parte del marcado de voz); ii) * no alineadas*, donde la entrada y el objetivo no muestran necesariamente una correspondencia paso a paso (por ejemplo, traducción automática).

Antes de preocuparnos por el manejo de objetivos de cualquier tipo, podemos abordar el problema más directo: modelado de densidad sin supervisión (también llamado modelado de secuencias*). Aquí, dada una colección de secuencias, nuestro objetivo es estimar la función de masa de probabilidad que nos dice lo probable que somos para ver cualquier secuencia dada, es decir, $p(\mathbf{x}_1, \ldots, \mathbf{x}_T)$.


In [ ]:
%matplotlib inline
import torch
from torch import nn
from laboratorio import d2l

## Modelos autorregresivos
Antes de introducir redes neuronales especializadas diseñadas para manejar datos secuencialmente estructurados, echemos un vistazo a algunos datos de secuencia reales y construyamos algunas intuiciones básicas y herramientas estadísticas. En particular, nos centraremos en los datos de precios de acciones del índice FTSE 100 ([Referencia fig_ftse100](https://d2l.ai/chapter_recurrent-neural-networks/sequence.html#fig-ftse100)). En cada paso *tiempo* $t \in \mathbb{Z}^+$, observamos el precio, $x_t$, del índice en ese momento.

![Índice FTSE 100 durante unos treinta años.](../recursos/originales/ftse100.png)

<a id="fig_ftse100"></a>

Ahora supongamos que un trader quisiera hacer operaciones a corto plazo, estratégicamente entrar o salir del índice, dependiendo de si creen que subirá o disminuirá en el paso de tiempo siguiente. Ausente cualquier otra característica (noticias, datos de información financiera, etc.), la única señal disponible para predecir el valor posterior es la historia de los precios hasta la fecha.

$$P(x_t \mid x_{t-1}, \ldots, x_1)$$

Si bien la estimación de toda la distribución sobre una variable aleatoria continuamente valorada puede ser difícil, el trader estaría encantado de centrarse en algunas estadísticas clave de la distribución, en particular el valor esperado y la varianza. Una estrategia simple para estimar la expectativa condicional

$$\mathbb{E}[(x_t \mid x_{t-1}, \ldots, x_1)],$$

Sería aplicar un modelo de regresión lineal (recordar [Referencia sec_linear_regression](https://d2l.ai/chapter_linear-regression/linear-regression.html#sec-linear-regression)). Tales modelos que revierten el valor de una señal en los valores anteriores de esa misma señal se llaman naturalmente *modelos autorregresivos*. Hay sólo un problema importante: el número de entradas, $x_{t-1}, \ldots, x_1$ varía, dependiendo de $t$. En otras palabras, el número de entradas aumenta con la cantidad de datos que encontramos. Así, si queremos tratar nuestros datos históricos como un conjunto de entrenamiento, nos quedamos con el problema de que cada ejemplo tiene un número diferente de características. Gran parte de lo que sigue en este capítulo girará en torno a técnicas para superar estos desafíos al involucrarnos en tales problemas de modelado *autoregresivos* donde el objeto de interés es $P(x_t \mid x_{t-1}, \ldots, x_1)$ o algunas estadísticas de esta distribución.

Algunas estrategias recurren con frecuencia. En primer lugar, podríamos creer que aunque las secuencias largas $x_{t-1}, \ldots, x_1$ están disponibles, puede que no sea necesario mirar hacia atrás en la historia a la hora de predecir el futuro próximo. En este caso podríamos contentarnos con condicionar en alguna ventana de longitud $\tau$ y sólo utilizar observaciones $x_{t-1}, \ldots, x_{t-\tau}$. El beneficio inmediato es que ahora el número de argumentos es siempre el mismo, al menos para $t > \tau$. Esto nos permite entrenar cualquier modelo lineal o red profunda que requiera vectores de longitud fija como entradas. En segundo lugar, podríamos desarrollar modelos que mantengan algún resumen $h_t$ de las observaciones pasadas (véase [Referencia fig_sequence-model](https://d2l.ai/chapter_recurrent-neural-networks/sequence.html#fig-sequence-model)) y al mismo tiempo actualizar $h_t$ además de la predicción $\hat{x}_t$. Esto conduce a modelos que estiman no sólo $x_t$ con $\hat{x}_t = P(x_t \mid h_{t})$ sino también actualizaciones de la forma $h_t = g(h_{t-1}, x_{t-1})$. Como $h_t$ nunca se observa, estos modelos también se llaman *modelos autorregresivos latentes*.

![Modelo autorregresivo latente.](../recursos/originales/sequence-model.svg)
<a id="fig_sequence-model"></a>

Para construir datos de entrenamiento a partir de datos históricos, uno típicamente crea ejemplos mediante ventanas de muestreo al azar. En general, no esperamos que el tiempo se detenga. Sin embargo, a menudo asumimos que mientras que los valores específicos de $x_t$ podrían cambiar, la dinámica según la cual se genera cada observación posterior dada las observaciones anteriores no lo hacen. Los estadísticos llaman dinámica que no cambia *estacionario*.

## Modelos de secuencia
A veces, especialmente al trabajar con el lenguaje, deseamos estimar la probabilidad conjunta de una secuencia completa. Esta es una tarea común cuando se trabaja con secuencias compuestas de *tokens* discretos, tales como palabras. Generalmente, estas funciones estimadas se llaman *modelos de secuencia* y para datos de lenguaje natural, se llaman *modelos de lenguaje*. El campo de modelado de secuencias ha sido impulsado tanto por el procesamiento de lenguaje natural, que a menudo describimos los modelos de secuencias como "modelos de lenguaje", incluso cuando se trata de datos no lingüísticos. Los modelos de lenguaje resultan útiles por todo tipo de razones. A veces queremos evaluar la probabilidad de oraciones. Por ejemplo, podríamos querer comparar la naturalidad de dos salidas candidatas generadas por un sistema de traducción automática o por un sistema de reconocimiento de voz. Pero el modelado de lenguaje nos da no sólo la capacidad de *evaluar* probabilidad, sino la capacidad de *sample* secuencias, e incluso optimizar para las secuencias más probables.

Aunque el modelado del lenguaje puede no parecer, a primera vista, un problema autorregresivo, podemos reducir el modelado del lenguaje a predicción autorregresiva al descomponer la densidad conjunta de una secuencia $p(x_1, \ldots, x_T)$ en el producto de densidades condicionales de una manera de izquierda a derecha aplicando la regla de la cadena de probabilidad:

$$P(x_1, \ldots, x_T) = P(x_1) \prod_{t=2}^T P(x_t \mid x_{t-1}, \ldots, x_1).$$

Tenga en cuenta que si estamos trabajando con señales discretas tales como palabras, entonces el modelo autorregresivo debe ser un clasificador probabilístico, la salida de una distribución de probabilidad completa sobre el vocabulario para cualquier palabra vendrá a continuación, dado el contexto a la izquierda.

### Modelos Markov
<a id="subsec_markov-models"></a>

Ahora supongamos que deseamos emplear la estrategia mencionada anteriormente, donde sólo condicionamos en los pasos de tiempo anteriores $\tau$, es decir, $x_{t-1}, \ldots, x_{t-\tau}$, en lugar de toda la historia de secuencia $x_{t-1}, \ldots, x_1$. Siempre que podemos tirar la historia más allá de los pasos anteriores $\tau$ sin ninguna pérdida de poder predictivo, decimos que la secuencia satisface una *condición de Markov*, es decir, *que el futuro es condicionalmente independiente del pasado, dada la historia reciente*. Cuando $\tau = 1$, decimos que los datos se caracterizan por un *modelo de Markov de primer orden*, y cuando $\tau = k$, decimos que los datos se caracterizan por un modelo de Markov de orden $k^{\textrm{th}}$. Para cuando la condición de Markov de primer orden mantiene ($\tau = 1$) la factorización de nuestra probabilidad conjunta se convierte en un producto de probabilidades de cada palabra dada la palabra anterior *word*:

$$P(x_1, \ldots, x_T) = P(x_1) \prod_{t=2}^T P(x_t \mid x_{t-1}).$$

A menudo nos resulta útil trabajar con modelos que proceden como si una condición de Markov se satisfizo, incluso cuando sabemos que esto es sólo * aproximadamente * verdad. Con los documentos de texto reales seguimos ganando información a medida que incluimos más y más contexto hacia la izquierda. Pero estas ganancias disminuyen rápidamente. Así, a veces comprometemos, obviando dificultades computacionales y estadísticas por modelos de formación cuya validez depende de una condición de Markov $k^{\textrm{th}}$ orden. Incluso los modelos masivos de lenguaje basados en RNN y Transformer rara vez incorporan más de miles de palabras de contexto.

Con datos discretos, un verdadero modelo de Markov simplemente cuenta el número de veces que cada palabra ha ocurrido en cada contexto, produciendo la estimación de frecuencia relativa de $P(x_t \mid x_{t-1})$. Siempre que los datos asumen sólo valores discretos (como en el lenguaje), la secuencia más probable de palabras se puede calcular eficientemente utilizando la programación dinámica.

### El orden de decodificación
Puede que se pregunte por qué representamos la factorización de una secuencia de texto $P(x_1, \ldots, x_T)$ como una cadena de probabilidades condicionales de izquierda a derecha. ¿Por qué no de derecha a izquierda o algún otro orden aparentemente aleatorio? En principio, no hay nada de malo en desplegar $P(x_1, \ldots, x_T)$ en orden inverso. El resultado es una factorización válida:

$$P(x_1, \ldots, x_T) = P(x_T) \prod_{t=T-1}^1 P(x_t \mid x_{t+1}, \ldots, x_T).$$

Sin embargo, hay muchas razones por las que se prefiere factorizar el texto en la misma dirección en la que lo leemos (de izquierda a derecha para la mayoría de los idiomas, pero de derecha a izquierda para el árabe y el hebreo) para la tarea de modelar el lenguaje. Primero, esta es sólo una dirección más natural en la que debemos pensar. Después de todo, todos leemos el texto todos los días, y este proceso se guía por nuestra capacidad de anticipar las palabras y frases que probablemente vendrán a continuación. Sólo piensa en cuántas veces has completado la oración de otra persona. Así, incluso si no tuviéramos otra razón para preferir tales decodificación en orden, serían útiles si sólo porque tenemos mejores intuiciones para lo que debería ser probable al predecir en este orden.

En segundo lugar, al factorizar en orden, podemos asignar probabilidades a secuencias arbitrariamente largas usando el mismo modelo de lenguaje. Para convertir una probabilidad sobre los pasos $1$ a $t$ en uno que se extiende a la palabra $t+1$ simplemente multiplicamos por la probabilidad condicional del token adicional dado los anteriores: $P(x_{t+1}, \ldots, x_1) = P(x_{t}, \ldots, x_1) \cdot P(x_{t+1} \mid x_{t}, \ldots, x_1)$.

En tercer lugar, tenemos modelos predictivos más fuertes para predecir palabras adyacentes que palabras en otros lugares arbitrarios. Si bien todos los órdenes de factorización son válidos, no necesariamente todos representan problemas de modelado predictivo igualmente fáciles. Esto es cierto no sólo para el lenguaje, sino también para otros tipos de datos, por ejemplo, cuando los datos se estructuran causalmente. Por ejemplo, creemos que los eventos futuros no pueden influir en el pasado. Por lo tanto, si cambiamos $x_t$, podemos ser capaces de influir en lo que sucede para $x_{t+1}$ en adelante pero no lo contrario. Es decir, si cambiamos $x_t$, la distribución sobre eventos pasados no cambiará. En algunos contextos, esto hace más fácil predecir $P(x_{t+1} \mid x_t)$ que predecir $P(x_t \mid x_{t+1})$. Por ejemplo, en algunos casos, podemos encontrar $x_{t+1} = f(x_t) + \epsilon$ para algún ruido aditivo $\epsilon$, mientras que lo contrario no es verdad [Hoyer.Janzing.Mooij.ea.2009](https://d2l.ai/chapter_references/zreferences.html). Esta es una gran noticia, ya que es típicamente la dirección hacia adelante que nos interesa estimar. El libro de [Peters.Janzing.Scholkopf.2017](https://d2l.ai/chapter_references/zreferences.html) contiene más sobre este tema.

## Entrenamiento
Antes de centrar nuestra atención en los datos de texto, primero probemos esto con algunos datos sintéticos de valor continuo.

** Aquí, nuestros 1000 datos sintéticos seguirán la función trigonométrica `sin`, aplicada a 0.01 veces el paso del tiempo. Para hacer el problema un poco más interesante, corrompemos cada muestra con ruido aditivo.** De esta secuencia extraemos ejemplos de entrenamiento, cada uno consistente en características y una etiqueta.


In [ ]:
class Data(d2l.DataModule):
    def __init__(self, batch_size=16, T=1000, num_train=600, tau=4):
        self.save_hyperparameters()
        self.time = torch.arange(1, T + 1, dtype=torch.float32)
        self.x = torch.sin(0.01 * self.time) + torch.randn(T) * 0.2

In [ ]:
data = Data()
d2l.plot(data.time, data.x, 'time', 'x', xlim=[1, 1000], figsize=(6, 3))

Para empezar, probamos un modelo que actúa como si los datos satisficieran una condición de Markov de orden $\tau^{\textrm{th}}$, y por lo tanto predice $x_t$ usando solamente las observaciones pasadas de $\tau$. **Así para cada paso del tiempo tenemos un ejemplo con la etiqueta $y  = x_t$ y características $\mathbf{x}_t = [x_{t-\tau}, \ldots, x_{t-1}]$.** El lector astuto podría haber notado que esto resulta en ejemplos $1000-\tau$, ya que carecemos de historia suficiente para $y_1, \ldots, y_\tau$. Si bien podríamos rellenar las primeras secuencias $\tau$ con ceros, para mantener las cosas simples, las dejamos caer por ahora. El conjunto de datos resultante contiene ejemplos $T - \tau$, donde cada entrada al modelo tiene longitud de secuencia $\tau$. Nosotros **creamos un iterador de datos en los primeros 600 ejemplos**, cubriendo un período de la función de pecado.


In [ ]:
@d2l.add_to_class(Data)
def get_dataloader(self, train):
    features = [self.x[i : self.T-self.tau+i] for i in range(self.tau)]
    self.features = torch.stack(features, 1)
    self.labels = self.x[self.tau:].reshape((-1, 1))
    i = slice(0, self.num_train) if train else slice(self.num_train, None)
    return self.get_tensorloader([self.features, self.labels], train, i)

En este ejemplo, nuestro modelo será una regresión lineal estándar.


In [ ]:
model = d2l.LinearRegression(lr=0.01)
trainer = d2l.Trainer(max_epochs=5)
trainer.fit(model, data)

### Nota docente de Hespérides

Escribe qué información puede ver cada posición. Una máscara causal impide consultar el futuro; una máscara de padding excluye posiciones que no son datos. Comprueba que cada fila de atención suma uno antes de aplicar dropout. Los mapas de atención describen mezclas de valores, pero por sí solos no prueban una explicación causal del modelo.

Vínculo con los apuntes: sesión 5, «Series y contexto».


## Predicción
** Para evaluar nuestro modelo, primero comprobamos lo bien que funciona en la predicción de un paso por delante**.


In [ ]:
onestep_preds = model(data.features).detach().numpy()
d2l.plot(data.time[data.tau:], [data.labels, onestep_preds], 'time', 'x',
         legend=['labels', 'Predicción a un paso'], figsize=(6, 3))

Estas predicciones se ven bien, incluso cerca del final en $t=1000$.

Pero, ¿y si sólo observamos datos de secuencia hasta el paso de tiempo 604 (`n_train + tau`) y deseamos hacer predicciones varios pasos en el futuro? Desafortunadamente, no podemos calcular directamente la predicción de un paso por delante para el paso de tiempo 609, porque no conocemos las entradas correspondientes, habiendo visto sólo hasta $x_{604}$. Podemos abordar este problema conectando nuestras predicciones anteriores como entradas a nuestro modelo para hacer predicciones posteriores, proyectando hacia adelante, un paso a la vez, hasta alcanzar el paso de tiempo deseado:

$$\begin{aligned}
\hat{x}_{605} &= f(x_{601}, x_{602}, x_{603}, x_{604}), \\
\hat{x}_{606} &= f(x_{602}, x_{603}, x_{604}, \hat{x}_{605}), \\
\hat{x}_{607} &= f(x_{603}, x_{604}, \hat{x}_{605}, \hat{x}_{606}),\\
\hat{x}_{608} &= f(x_{604}, \hat{x}_{605}, \hat{x}_{606}, \hat{x}_{607}),\\
\hat{x}_{609} &= f(\hat{x}_{605}, \hat{x}_{606}, \hat{x}_{607}, \hat{x}_{608}),\\
&\vdots\end{aligned}$$

Generalmente, para una secuencia observada $x_1, \ldots, x_t$, su previsión de salida $\hat{x}_{t+k}$ en el momento del paso $t+k$ se llama la predicción $k$*-step-ahead*. Ya que hemos observado hasta $x_{604}$, su predicción $k$-step-ahead es $\hat{x}_{604+k}$. En otras palabras, tendremos que seguir usando nuestras propias predicciones para hacer predicciones multistep-ahead. Veamos lo bien que va esto.


In [ ]:
multistep_preds = torch.zeros(data.T)
multistep_preds[:] = data.x
for i in range(data.num_train + data.tau, data.T):
    multistep_preds[i] = model(
        multistep_preds[i - data.tau:i].reshape((1, -1)))
multistep_preds = multistep_preds.detach().numpy()

In [ ]:
d2l.plot([data.time[data.tau:], data.time[data.num_train+data.tau:]],
         [onestep_preds, multistep_preds[data.num_train+data.tau:]], 'time',
         'x', legend=['Predicción a un paso', 'Predicción recursiva'], figsize=(6, 3))

Por desgracia, en este caso fallamos espectacularmente. Las predicciones decaen a una constante bastante rápidamente después de unos pocos pasos. ¿Por qué el algoritmo funciona tanto peor al predecir más adelante en el futuro? En última instancia, esto se debe al hecho de que los errores se acumulan. Digamos que después del paso 1 tenemos algún error $\epsilon_1 = \bar\epsilon$. Ahora el *input* para el paso 2 está perturbado por $\epsilon_1$, por lo que sufrimos algún error en el orden de $\epsilon_2 = \bar\epsilon + c \epsilon_1$ para alguna constante $c$, y así sucesivamente. Las predicciones pueden divergir rápidamente de las observaciones verdaderas. Puede que ya esté familiarizado con este fenómeno común. Por ejemplo, las previsiones meteorológicas para las próximas 24 horas tienden a ser bastante precisos pero más allá de eso, la precisión disminuye rápidamente. Discutiremos métodos para mejorar esto a lo largo de este capítulo y más allá.

Echemos un vistazo más de cerca a las dificultades en las predicciones de $k$-step-ahead** computando predicciones sobre toda la secuencia para $k = 1, 4, 16, 64$.


In [ ]:
def k_step_pred(k):
    features = []
    for i in range(data.tau):
        features.append(data.x[i : i+data.T-data.tau-k+1])
    # El elemento (i+tau)-th almacena las predicciones (i+1)-paso por delante
    for i in range(k):
        preds = model(torch.stack(features[i : i+data.tau], 1))
        features.append(preds.reshape(-1))
    return features[data.tau:]

In [ ]:
steps = (1, 4, 16, 64)
preds = k_step_pred(steps[-1])
d2l.plot(data.time[data.tau+steps[-1]-1:],
         [preds[k - 1].detach().numpy() for k in steps], 'time', 'x',
         legend=[f'{k}-step preds' for k in steps], figsize=(6, 3))

Esto ilustra claramente cómo la calidad de la predicción cambia a medida que tratamos de predecir más adelante en el futuro. Mientras que las predicciones de 4 pasos por delante todavía se ven bien, cualquier cosa más allá de eso es casi inútil.

## Resumen
Hay una gran diferencia en la dificultad entre interpolación y extrapolación. En consecuencia, si usted tiene una secuencia, siempre respetar el orden temporal de los datos cuando se entrena, es decir, nunca entrenar en datos futuros. Dado este tipo de datos, los modelos de secuencia requieren herramientas estadísticas especializadas para la estimación. Dos opciones populares son los modelos autorregresivos y los modelos autorregresivos latentes variables. Para los modelos causales (por ejemplo, el tiempo que va hacia adelante), la estimación de la dirección hacia adelante es típicamente mucho más fácil que la dirección inversa. Para una secuencia observada hasta el paso $t$ del tiempo, su salida predicha en el paso $t+k$ es la predicción $k$*-paso-ahead*. Como predicemos con el tiempo aumentando $k$, los errores se acumulan y la calidad de la predicción se degrada, a menudo dramáticamente.

## Ejercicios
1. Mejorar el modelo en el experimento de esta sección.
    1. ¿Incorpore más de las últimas cuatro observaciones? ¿Cuántas necesita realmente?
    1. ¿Cuántas observaciones pasadas necesitarías si no hubiera ruido? Consejo: puedes escribir $\sin$ y $\cos$ como una ecuación diferencial.
    1. ¿Puede incorporar observaciones antiguas manteniendo constante el número total de características? ¿Mejora la precisión? ¿Por qué?
    1. Cambia la arquitectura de red neuronal y evalúa el rendimiento. Puedes entrenar el nuevo modelo con más épocas. ¿Qué observas?
1. Un inversor quiere encontrar una buena seguridad para comprar. Miran los rendimientos pasados para decidir cuál es probable que hacer bien. ¿Qué podría ir mal con esta estrategia?
1. ¿Se aplica también la causalidad al texto? ¿Hasta qué punto?
1. Dé un ejemplo para cuando un modelo autorregresivo latente podría ser necesario para capturar la dinámica de los datos.


[Debate del original](https://discuss.d2l.ai/t/114)
